# Journey mapping graph

In this section, we simply choose which one of the two graphs we want to work with, and using graphviz,display the information it contains, namely:
- a node is a tuple `(station ID, station name, sequence)`, and has for data a dictionary with the three booleans `{is_origin, is_destination, is_matched}`
- the edges are directional, and contain for data the boolean `walk`, plus if its false the sets of `route_names` and `route_ids`


In [16]:
import pickle

graph = pickle.load(open('small_graph.pickle', 'rb'))

In [17]:
from graphviz import Digraph
from re import sub
from typing import Any, Optional
from IPython.display import display, SVG, HTML

def node_name(node: tuple[str, str, int], data: dict) -> str:
    return f"{node[1]}\n(id={node[0]}, seq={node[2]})"

def node_attrs(data: dict) -> dict:
    if data["is_origin"]:
        return dict(fillcolor="green4", style="filled", fontcolor="white")
    if data["is_destination"]:
        return dict(fillcolor="gold2", style="filled")
    if data["is_matched"]:
        return dict(fillcolor="lightskyblue2", style="filled")
    return dict()

g = Digraph()
g.node_attr["fontname"] = "sans-serif"

for a, data in graph.nodes(data=True):
    g.node(node_name(a, data), label=node_name(a, data), **node_attrs(data))

for a, b, edge_data in graph.edges(data=True):
    edge_style = "dashed" if "walk" in edge_data else "" 
    edge_label = "walk" if "walk" in edge_data else ", ".join(edge_data.get("route_names", []))
    g.edge(node_name(a, graph.nodes.get(a)), node_name(b,graph.nodes.get(b)), label=edge_label, style=edge_style)

g.attr(size="15,15")

# Display as SVG with scrollbars
display(HTML(f'''
<div style="height:500px; overflow:scroll; border:1px solid black;">
    {g.pipe(format='svg').decode('utf-8')}
</div>
'''))

# Find shortest paths

You'll find below a short example of how to use the function from networkx to find shortest paths:
- `shortest_path` returns the one shortest path between two nodes
- `shortest_simple_paths` returns an iterator that gives all paths from shortest to longest

**Question**
Can you propose a way to use it (changing the nodes used, the cost function, ...), to make sure the true path is returned in the set of results?

*(the true path we're looking for are indicated in the json file, you can see them loaded in the last cell)*

In [18]:
from toolz import sliding_window
from networkx import shortest_path, shortest_simple_paths, exception

def cost_function(u, v, data) -> int:
    return 1

# Get one single path
source = next(node for node, data in graph.nodes(data=True) if data["is_origin"])
target = next(node for node, data in graph.nodes(data=True) if data["is_destination"])
path = shortest_path(graph, source, target, cost_function)

print(f"The shortest path between {source} and {target} has {len(path)} nodes")

# Gather multiple paths
all_paths = []
try:
    for path in shortest_simple_paths(graph, source, target, cost_function):
        all_paths.append(path)
except exception.NetworkXNoPath:
    pass

print(f"\nIn total, {len(all_paths)} possible paths betweeen {source} and {target}")

The shortest path between ('8501094', 'Apples', 0) and ('8501037', 'Morges', 0) has 11 nodes

In total, 2 possible paths betweeen ('8501094', 'Apples', 0) and ('8501037', 'Morges', 0)


In [19]:
import json
from pprint import pprint

with open('expected_paths.json', 'r') as f:
    expected_paths = json.load(f)

print("Best path to find on the graph:\n")
pprint([tuple(node) for node in expected_paths["small_graph"]])

Best path to find on the graph:

[('8501094', 'Apples', 0),
 ('8501083', 'Reverolle', 0),
 ('8501082', 'Chardonney-Château', 0),
 ('8501093', 'Yens', 0),
 ('8501092', 'Bussy-Chardonney', 0),
 ('8501080', 'Le Marais', 0),
 ('8501091', 'Vufflens-le-Château', 0),
 ('8501081', 'Chigny', 0),
 ('8501090', 'Prélionne', 0),
 ('8501054', 'La Gottaz', 0),
 ('8501037', 'Morges', 0)]


# Analyzing the results

**Question**

If you take the 100 shortest paths on the large graph, what kind of analytics can you draw for them? (use the large graph for this)

In [20]:
import pandas as pd
import matplotlib.pyplot as plt

